# Randomized smoothing

This Python script trains a Convolutional Neural Network (CNN) on the CIFAR-10 dataset with and without randomized smoothing. The goal of randomized smoothing is to improve the robustness of the model against adversarial noise by training and evaluating it with added Gaussian noise.

This setup loads CIFAR-10 and prepares it for a randomised smoothing demo. After importing TensorFlow/Keras, the dataset of 60,000 colour images across 10 classes is loaded and split into training and test sets. Pixel values are normalised to the range [0, 1] to stabilise optimisation, and labels are converted to one-hot vectors with 10 entries to match a softmax classifier’s output layer. This minimal preprocessing provides a clean baseline before introducing noise injection and certification steps used in randomised smoothing.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

# Load and preprocess CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

This function defines a straightforward convolutional neural network to serve as the base classifier in the randomised smoothing experiment. The network begins with two convolutional layers (32 and 64 filters, 3×3 kernels, ReLU activations) each followed by max pooling to downsample feature maps and retain salient patterns. The output is flattened and passed through a fully connected layer of 64 ReLU units before reaching a 10-way softmax layer for classification across CIFAR-10 classes. Compiled with the Adam optimiser, categorical cross-entropy loss, and accuracy as a metric, this simple CNN provides a baseline model whose predictions can later be stabilised and certified against adversarial perturbations through the use of randomised smoothing.

In [ ]:
# Define a simple CNN model
def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

This function introduces Gaussian noise to implement the core mechanism of randomised smoothing. For each input image, noise is sampled from a normal distribution with mean 0 and a configurable standard deviation (stddev), which controls the strength of the perturbation. The noisy images are then clipped to remain within the valid pixel range [0, 1]. By repeatedly evaluating the model on such noisy versions of an input, randomised smoothing produces predictions that are less sensitive to small adversarial perturbations, enabling both improved empirical robustness and the possibility of formal robustness guarantees.

In [ ]:
# Add noise for randomized smoothing
def add_noise(inputs, stddev=0.1):
    """
    Adds Gaussian noise to the inputs.
    :param inputs: Input images
    :param stddev: Standard deviation of the Gaussian noise
    :return: Noisy inputs
    """
    noise = tf.random.normal(shape=tf.shape(inputs), mean=0.0, stddev=stddev)
    return tf.clip_by_value(inputs + noise, 0.0, 1.0)  # Ensure values remain in [0, 1]

This function trains the CNN using randomised smoothing, where Gaussian noise is deliberately injected into the training data to improve robustness. For each epoch, the training set is processed in batches of a specified size. Within each batch, input images are perturbed by adding noise sampled from a normal distribution with the chosen standard deviation (stddev). The model is then trained directly on these noisy images using train_on_batch, encouraging it to learn stable decision boundaries that are less sensitive to small input perturbations. By repeating this process across multiple epochs, the model develops smoother predictions, forming the foundation for robustness certification under randomised smoothing.

In [ ]:
# Train the model with randomized smoothing
def train_with_smoothing(model, x_train, y_train, epochs=5, batch_size=32, stddev=0.1):
    """
    Trains the model with randomized smoothing.
    :param model: The base model
    :param x_train: Training data
    :param y_train: Training labels
    :param epochs: Number of epochs
    :param batch_size: Batch size
    :param stddev: Standard deviation of the noise
    """
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        for i in range(0, len(x_train), batch_size):
            x_batch = x_train[i:i + batch_size]
            y_batch = y_train[i:i + batch_size]

            # Add Gaussian noise to the inputs
            noisy_x_batch = add_noise(x_batch, stddev=stddev)

            # Train on noisy data
            model.train_on_batch(noisy_x_batch, y_batch)

This function evaluates the robustness of the trained model under randomised smoothing by averaging predictions across multiple noisy versions of the test data. For each of the specified number of samples (num_samples), Gaussian noise is added to the test inputs, and the model generates predictions. These predictions are collected and then averaged element-wise across all noisy trials, producing a “smoothed” probability distribution for each input. The final class label is chosen as the one with the highest average probability, and accuracy is computed by comparing these labels against the true test labels. By aggregating predictions in this way, the evaluation measures how consistently the model classifies inputs under random perturbations—demonstrating its empirical robustness and laying the groundwork for formal certification of guaranteed accuracy within a noise radius.

In [ ]:
# Evaluate the model with randomized smoothing
def evaluate_with_smoothing(model, x_test, y_test, num_samples=10, stddev=0.1):
    """
    Evaluates the model with randomized smoothing by averaging predictions over noisy samples.
    :param model: The trained model
    :param x_test: Test data
    :param y_test: Test labels
    :param num_samples: Number of noisy samples to average over
    :param stddev: Standard deviation of the noise
    :return: Accuracy
    """
    smoothed_predictions = []

    for i in range(num_samples):
        noisy_x_test = add_noise(x_test, stddev=stddev)
        predictions = model.predict(noisy_x_test)
        smoothed_predictions.append(predictions)

    # Average the predictions over all noisy samples
    averaged_predictions = np.mean(smoothed_predictions, axis=0)
    predicted_labels = np.argmax(averaged_predictions, axis=1)
    true_labels = np.argmax(y_test, axis=1)

    accuracy = np.mean(predicted_labels == true_labels)
    print(f"Smoothed Accuracy: {accuracy * 100:.2f}%")
    return accuracy

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 17s 0us/step
Training the model with randomized smoothing...
Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 2/5
Epoch 3/5
Epoch 4/5
Epoch 5/5
Evaluating the model with randomized smoothing...
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step
Smoothed Accuracy: 65.65%


0.6565

We test the model without noise, and then with noise applied.

In [ ]:
# Train and evaluate the model
print("Training the model with randomized smoothing...")
model = create_model()
train_with_smoothing(model, x_train, y_train, epochs=5, stddev=0.1)

print("Evaluating the model with randomized smoothing...")
evaluate_with_smoothing(model, x_test, y_test, num_samples=10, stddev=0.1)

This final stage compares the model’s performance on clean test data versus its performance under randomised smoothing. The evaluate_without_smoothing function measures baseline accuracy directly on the unmodified test set. In contrast, the main evaluation flow also calls evaluate_with_smoothing, which averages predictions across multiple noisy versions of the inputs to estimate smoothed accuracy. Printing both results side by side highlights the trade-off: accuracy on clean data may be slightly lower, but smoothed accuracy reflects the model’s improved stability when faced with noisy or perturbed inputs. This demonstrates the central idea of randomised smoothing—sacrificing a small amount of clean accuracy in exchange for stronger robustness against adversarial or unexpected variations.

In [ ]:
# Evaluate the model without smoothing
def evaluate_without_smoothing(model, x_test, y_test):
    """
    Evaluates the model on the original test data without adding noise.
    :param model: The trained model
    :param x_test: Test data
    :param y_test: Test labels
    :return: Accuracy
    """
    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
    print(f"Original Accuracy: {accuracy * 100:.2f}%")
    return accuracy

# Main evaluation flow
print("Evaluating the model without smoothing...")
original_accuracy = evaluate_without_smoothing(model, x_test, y_test)

print("Evaluating the model with randomized smoothing...")
smoothed_accuracy = evaluate_with_smoothing(model, x_test, y_test, num_samples=10, stddev=0.1)

print(f"Original Accuracy: {original_accuracy * 100:.2f}%")
print(f"Smoothed Accuracy: {smoothed_accuracy * 100:.2f}%")


Evaluating the model without smoothing...
Original Accuracy: 63.15%
Evaluating the model with randomized smoothing...
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
Smoothed Accuracy: 65.54%
Original Accuracy: 63.15%
Smoothed Accuracy: 65.54%
